# Model Comparison and Threshold Selection

## Business question

Which baseline model performs better for high-risk prioritization, and which decision threshold best matches review capacity?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

from cx_complaints.features.text_features import add_text_features
from cx_complaints.modelling.baselines import (
    baseline_metrics_table,
    run_text_baseline_models,
)
from cx_complaints.modelling.risk import score_complaint_risk
from cx_complaints.modelling.supervised import time_aware_train_test_split
from cx_complaints.preprocessing.text import add_clean_text_column

plt.style.use("ggplot")

## 1. Load and prepare modeling dataset

In [ ]:
raw_path = Path("data/raw/cfpb_sample.csv")
if not raw_path.exists():
    raise FileNotFoundError(
        "Expected data/raw/cfpb_sample.csv. Run the fetch command first."
    )

data = pd.read_csv(raw_path)
data = add_clean_text_column(data)
data = add_text_features(data)
data = score_complaint_risk(data)
data["target_high_risk"] = (data["risk_level"] == "high").astype(int)

split = time_aware_train_test_split(data, date_column="date_received", train_fraction=0.8)
train = split.train.copy()
test = split.test.copy()

print(f"Train rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"Split date: {split.split_timestamp.date()}")

## 2. Compare baseline models

In [ ]:
baseline_results = run_text_baseline_models(
    train_frame=train,
    test_frame=test,
    text_column="clean_text",
    target_column="target_high_risk",
    min_df=3,
    max_features=4000,
)

metrics_table = baseline_metrics_table(baseline_results)
metrics_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
metrics_long = metrics_table.melt(
    id_vars=["model_name"],
    value_vars=["precision", "recall", "f1"],
    var_name="metric",
    value_name="value",
)

for metric in ["precision", "recall", "f1"]:
    subset = metrics_long[metrics_long["metric"] == metric]
    ax.plot(subset["model_name"], subset["value"], marker="o", label=metric)

ax.set_ylim(0, 1)
ax.set_title("Baseline Model Comparison")
ax.set_ylabel("Score")
ax.legend()
fig.tight_layout()

## 3. Threshold tuning (logistic regression)

In [ ]:
vectorizer = TfidfVectorizer(min_df=3, max_features=4000)
x_train = vectorizer.fit_transform(train["clean_text"].fillna(""))
x_test = vectorizer.transform(test["clean_text"].fillna(""))

y_train = train["target_high_risk"].astype(int)
y_test = test["target_high_risk"].astype(int)

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(x_train, y_train)
test_scores = log_reg.predict_proba(x_test)[:, 1]

In [ ]:
thresholds = [round(value, 2) for value in np.arange(0.1, 0.95, 0.05)]
rows: list[dict[str, float]] = []
for threshold in thresholds:
    pred = (test_scores >= threshold).astype(int)
    rows.append(
        {
            "threshold": threshold,
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "flagged_complaints": int(pred.sum()),
            "flagged_share_pct": float(pred.mean() * 100),
        }
    )

threshold_table = pd.DataFrame(rows)
threshold_table

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(
    threshold_table["threshold"],
    threshold_table["precision"],
    marker="o",
    label="Precision",
)
ax1.plot(
    threshold_table["threshold"],
    threshold_table["recall"],
    marker="s",
    label="Recall",
)
ax2.bar(
    threshold_table["threshold"],
    threshold_table["flagged_share_pct"],
    alpha=0.2,
    width=0.03,
    label="Flagged share (%)",
)

ax1.set_xlabel("Threshold")
ax1.set_ylabel("Precision / Recall")
ax2.set_ylabel("Flagged share (%)")
ax1.set_title("Threshold Tradeoff for High-Risk Queue")
ax1.set_ylim(0, 1)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
fig.tight_layout()

## 4. Recommended operating threshold

Choose the threshold that balances:

1. acceptable analyst queue size (`flagged_share_pct`),
2. minimum precision target,
3. recall needed to avoid missing severe complaints.

In [ ]:
# Example: pick the highest recall while keeping precision >= 0.60 and flagged share <= 40%
candidate = threshold_table[
    (threshold_table["precision"] >= 0.60)
    & (threshold_table["flagged_share_pct"] <= 40)
]

if candidate.empty:
    print("No threshold met the example constraints.")
else:
    candidate.sort_values(["recall", "precision"], ascending=[False, False]).head(1)